# 15 - Word2Vec with Negative Sampling

Goal: Implement Word2Vec efficiently using negative sampling

Run with: conda activate tfenv

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import plotly.express as px
import pandas as pd
import time

print(f"TensorFlow version: {tf.__version__}")

I0000 00:00:1779046156.988542    8528 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779046157.319694    8528 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779046160.343982    8528 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow version: 2.21.0


In [5]:
# Importar embeddings Word2Vec entrenados previamente
target_embeddings = np.load('myWord2Vec/target_embeddings.npy')
context_embeddings = np.load('myWord2Vec/context_embeddings.npy')
print('Embeddings cargados:', target_embeddings.shape, context_embeddings.shape)

Embeddings cargados: (807, 64) (807, 64)


In [49]:
# Load dataset using HuggingFace datasets API (as in notebook 14)
print("Loading gaianet/london dataset...")
from datasets import load_dataset

ds = load_dataset("gaianet/london", split="train")
print(f"Dataset size: {len(ds)}")
texts = [row['text'] if 'text' in row else row.get('content', '') for row in ds][:10000]
full_text = ' '.join(texts[:5000])
print(f"Total chars: {len(full_text)}")

# Build vocabulary and preprocess words as in notebook 14
words = full_text.lower().split()
words = [w.strip('.,;:!?()[]\"\'-0123456789') for w in words]
words = [w for w in words if len(w) > 2]
print(f"Total words: {len(words)}")
print(f"Sample: {words[:20]}")

Loading gaianet/london dataset...
Dataset size: 661
Total chars: 186582
Total words: 23671
Sample: ['london', 'the', 'capital', 'and', 'largest', 'city', 'england', 'and', 'the', 'united', 'kingdom', 'with', 'population', 'around', 'million', 'and', 'the', 'largest', 'city', 'western']


In [50]:
# Build vocabulary (threshold 5 as in notebook 14)
from collections import Counter
word_counts = Counter(words)
vocab = [w for w, c in word_counts.items() if c >= 5]
text_vocab = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in text_vocab.items()}
text_vocab_size = len(text_vocab)
text_embed_dim = 64
print(f"Vocabulary: {text_vocab_size} words")

Vocabulary: 807 words


In [51]:
# Create training pairs (Skip-gram)
def create_pairs(words, window=2):
    pairs = []
    for i, word in enumerate(words):
        if word not in text_vocab:
            continue
        for j in range(max(0, i - window), min(len(words), i + window + 1)):
            if i != j and words[j] in text_vocab:
                pairs.append((text_vocab[word], text_vocab[words[j]]))
    return pairs

pairs = create_pairs(words)
train_words = np.array([p[0] for p in pairs], dtype=np.int32)
train_context = np.array([p[1] for p in pairs], dtype=np.int32)
print(f"Training pairs: {len(pairs)}")

Training pairs: 54998


In [52]:
# Generate negative samples
NEGATIVE_SAMPLING = 5

all_indices = np.arange(text_vocab_size)

np.random.seed(42)
negative_samples = []

for target_idx in train_context:
    neg_indices = np.random.choice(all_indices, size=NEGATIVE_SAMPLING, replace=False)
    for neg_idx in neg_indices:
        negative_samples.append((target_idx, neg_idx))

neg_words = np.array([p[0] for p in negative_samples], dtype=np.int32)
neg_context = np.array([p[1] for p in negative_samples], dtype=np.int32)
neg_labels = np.zeros(len(negative_samples), dtype=np.float32)

print(f"NEGATIVE_SAMPLING = {NEGATIVE_SAMPLING}")
print(f"Generated {len(negative_samples)} negative samples")

NEGATIVE_SAMPLING = 5
Generated 274990 negative samples


In [53]:
# Combine positive and negative samples
pos_labels = np.ones(len(train_words), dtype=np.float32)

all_words = np.concatenate([train_words, neg_words])
all_context = np.concatenate([train_context, neg_context])
all_labels = np.concatenate([pos_labels, neg_labels])

print(f"Total training samples: {len(all_words)}")
print(f"  Positive (1): {len(pos_labels)}")
print(f"  Negative (0): {len(neg_labels)}")

Total training samples: 329988
  Positive (1): 54998
  Negative (0): 274990


In [54]:
# Word2Vec with Negative Sampling - Custom Model
class Word2VecNS(keras.Model):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.target_embedding = layers.Embedding(
            vocab_size, embed_dim, name='embedding_target'
        )
        self.context_embedding = layers.Embedding(
            vocab_size, embed_dim, name='embedding_context'
        )
        # No Dense layer here, output is scalar logit
    def call(self, inputs):
        target, context = inputs
        target_emb = self.target_embedding(target)  # (batch, embed_dim)
        context_emb = self.context_embedding(context)  # (batch, embed_dim)
        dot_product = tf.reduce_sum(target_emb * context_emb, axis=-1, keepdims=True)  # (batch, 1)
        return tf.sigmoid(dot_product)  # (batch, 1)

model = Word2VecNS(text_vocab_size, text_embed_dim)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

print("Word2Vec with Negative Sampling model ready!")
model.summary()

Word2Vec with Negative Sampling model ready!


Model: "word2_vec_ns_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_target (Embedding)    │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_context (Embedding)   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [55]:
# Training with negative sampling
train_ds = tf.data.Dataset.from_tensor_slices(((all_words, all_context), all_labels))
train_ds = train_ds.shuffle(buffer_size=len(all_words)).batch(256).prefetch(tf.data.AUTOTUNE)

print("Training Word2Vec with Negative Sampling...")
start = time.time()

# model.fit expects a tuple (inputs, labels) for custom models
history = model.fit(
    train_ds,
    epochs=20,
    verbose=1
)

print(f"Training time: {time.time() - start:.2f}s")

Training Word2Vec with Negative Sampling...
Epoch 1/20


1290/1290 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8747 - loss: 0.3397
Epoch 2/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9085 - loss: 0.2357
Epoch 3/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9213 - loss: 0.2041
Epoch 4/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9253 - loss: 0.1964
Epoch 5/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9273 - loss: 0.1932
Epoch 6/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9276 - loss: 0.1925
Epoch 7/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9283 - loss: 0.1912
Epoch 8/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9284 - loss: 0.1911
Epoch 9/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9286 - loss: 0.1911
Epoch 10/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9283 - loss: 0.1922
Epoch 11/20
1290/1290 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9287 - loss: 0.1904
Epoch 12/20
1290/1290 ━━━━━━━━━━━━━━━━━━━

In [56]:
# Get embeddings (use target embedding)
target_embeddings = model.target_embedding.get_weights()[0]
context_embeddings = model.context_embedding.get_weights()[0]

final_embeddings = (target_embeddings + context_embeddings) / 2

print(f"Target embeddings shape: {target_embeddings.shape}")
print(f"Context embeddings shape: {context_embeddings.shape}")

Target embeddings shape: (807, 64)
Context embeddings shape: (807, 64)


In [57]:
# Find similar words
from numpy.linalg import norm

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b) + 1e-8)

def find_similar(word, top_n=5):
    if word not in text_vocab:
        return []
    word_idx = text_vocab[word]
    word_emb = final_embeddings[word_idx]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w != word:
            sim = cosine_similarity(word_emb, final_embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]

print("=== Similar Words (Negative Sampling) ===")
test_words = ['tecnología', 'empresas', 'datos', 'desarrollo', 'aprendizaje', 'software']
for word in test_words:
    if word in text_vocab:
        similar = find_similar(word)
        print(f"'{word}' -> {[(w, f'{s:.3f}') for w, s in similar]}")

=== Similar Words (Negative Sampling) ===


In [58]:
# Test analogies
def analogy(word_a, word_b, word_c):
    if word_a not in text_vocab or word_b not in text_vocab or word_c not in text_vocab:
        return None
    
    vec = final_embeddings[text_vocab[word_b]] - final_embeddings[text_vocab[word_a]] + final_embeddings[text_vocab[word_c]]
    
    similarities = []
    for w, idx in text_vocab.items():
        if w not in [word_a, word_b, word_c]:
            sim = cosine_similarity(vec, final_embeddings[idx])
            similarities.append((w, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[0]

print("=== Analogies ===")
result = analogy('inteligencia', 'artificial', 'aprendizaje')
if result:
    print(f"'inteligencia' - 'artificial' + 'aprendizaje' = '{result[0]}' ({result[1]:.3f})")

result = analogy('empresas', 'tecnología', 'datos')
if result:
    print(f"'empresas' - 'tecnología' + 'datos' = '{result[0]}' ({result[1]:.3f})")

=== Analogies ===


In [59]:
# Visualize in 3D with PCA
from sklearn.decomposition import PCA

top_words = [w for w, c in word_counts.most_common(200) if w in text_vocab]
top_indices = [text_vocab[w] for w in top_words]
top_embeddings = final_embeddings[top_indices]

pca = PCA(n_components=3)
embeddings_3d = pca.fit_transform(top_embeddings)

df = pd.DataFrame(embeddings_3d, columns=['x', 'y', 'z'])
df['word'] = top_words

fig = px.scatter_3d(df, x='x', y='y', z='z', text='word',
                 title='Word2Vec with Negative Sampling - Embeddings (PCA 3D)')
fig.update_traces(marker=dict(size=6), textposition='top center')
fig.update_layout(height=600, width=800)
fig.show()

In [60]:
# Summary
print("=== Summary ===")
print(f"Vocab size: {text_vocab_size}")
print(f"Embedding dim: {text_embed_dim}")
print(f"Positive pairs: {len(train_words)}")
print(f"Negative samples: {len(neg_labels)}")
print(f"Total training: {len(all_words)}")
print(f"Negative sampling ratio: {NEGATIVE_SAMPLING}:1")
print("\nModel uses:")
print("  - Two embeddings: target and context")
print("  - Dot product + sigmoid (binary classification)")
print("  - Much faster than full softmax!")

=== Summary ===
Vocab size: 807
Embedding dim: 64
Positive pairs: 54998
Negative samples: 274990
Total training: 329988
Negative sampling ratio: 5:1

Model uses:
  - Two embeddings: target and context
  - Dot product + sigmoid (binary classification)
  - Much faster than full softmax!
